In [0]:
sc

SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

In [0]:
from pyspark.sql.types import StringType,StructField, StringType, DateType, IntegerType
import pyspark.sql.functions as F

In [0]:


schema = StructType([
    StructField("id",IntegerType(),True),
    StructField("first_name",StringType(),True),
    StructField("last_name",StringType(),True),
    StructField("email",StringType(),True),
    StructField("gender",StringType(),True),
    StructField("ssn",StringType(),True),
    StructField("inserted_date",DateType(),True)
])


In [0]:
user_df = spark.read.format('csv').option('header','true').load("dbfs:/FileStore/shared_uploads/kk285507@outlook.com/user_data.csv")

In [0]:
user_df.display()

id,first_name,last_name,email,gender,ssn,inserted_date
1,Odetta,Aylett,oaylett0@twitpic.com,Female,203-14-6684,11/22/2024
2,Antonietta,McGourty,amcgourty1@nih.gov,Female,863-90-1571,02/23/2024
3,Javier,Branthwaite,jbranthwaite2@google.com,Male,706-58-8773,09/24/2024
4,Sydelle,Barenski,sbarenski3@youku.com,Female,899-60-7100,09/17/2024
5,Emlyn,Ferandez,eferandez4@t-online.de,Male,685-46-8896,06/01/2024
6,Dino,Kynman,dkynman5@wiley.com,Male,758-28-5682,04/24/2024
7,Joelynn,Randerson,jranderson6@nyu.edu,Female,231-04-1761,12/14/2024
8,Yurik,Dunster,ydunster7@amazon.co.uk,Genderqueer,807-48-2811,10/29/2024
9,Diann,Beaver,dbeaver8@psu.edu,Female,759-05-6633,10/23/2024
10,Winston,Puffett,wpuffett9@over-blog.com,Male,530-42-1726,06/03/2024


In [0]:
user_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- ssn: string (nullable = true)
 |-- inserted_date: string (nullable = true)



# Date Format conversion

In [0]:
user_df = user_df.withColumn("inserted_date",F.date_format(F.to_date(F.col("inserted_date"),'MM/dd/yyyy'),'yyyy-MM-dd'))

# Type Conversion

In [0]:
user_df = user_df.select([F.col(column.name).cast(column.dataType) for column in schema.fields])

In [0]:
user_df.select([F.sum(F.when(F.col(column).isNull(),1).otherwise(0)).alias(column) for column in user_df.columns]).show()


+---+----------+---------+-----+------+---+-------------+
| id|first_name|last_name|email|gender|ssn|inserted_date|
+---+----------+---------+-----+------+---+-------------+
|  0|         0|        0|    0|     0|  0|            0|
+---+----------+---------+-----+------+---+-------------+



In [0]:
user_df.orderBy(F.col("inserted_date").desc()).display()

id,first_name,last_name,email,gender,ssn,inserted_date
18,Tani,Swannick,tswannickh@studiopress.com,Female,773-49-1332,2025-01-20
98,Henry,Bygate,hbygate2p@arstechnica.com,Male,863-59-9997,2025-01-17
49,Marci,Quelch,mquelch1c@aol.com,Female,598-09-0176,2025-01-15
59,Ketti,Stronghill,kstronghill1m@vk.com,Non-binary,330-21-6946,2025-01-12
54,Sara,Nern,snern1h@typepad.com,Genderfluid,732-79-8205,2025-01-11
35,Kermit,McGeraghty,kmcgeraghtyy@seesaa.net,Male,262-21-3682,2025-01-05
40,Franklyn,Kedwell,fkedwell13@behance.net,Male,641-02-3450,2025-01-05
28,Bruis,Gutherson,bguthersonr@microsoft.com,Male,746-66-2137,2024-12-30
31,Eduino,Sciusscietto,esciussciettou@google.es,Male,249-22-8317,2024-12-22
82,Osborne,Laver,olaver29@slate.com,Male,493-82-5639,2024-12-21


In [0]:
new_user_df = spark.read.format("csv").option("header", "true").schema(schema).load("dbfs:/FileStore/shared_uploads/kk285507@outlook.com/new_data.csv")

In [0]:
new_user_df.display()

id,first_name,last_name,email,gender,ssn,inserted_date
18,Tani,Swannick,tswannickh@studio...,Female,773-49-1332,2025-02-20
98,Henry,Bygate,hbygate2p@arstech...,Male,863-59-9997,2025-02-17
49,Marci,Quelch,mquelch1c@aol.com,Female,598-09-0176,2025-02-15
59,Ketti,Stronghill,kstronghill1m@vk.com,Non-binary,330-21-6946,2025-02-12
54,Sara,Nern,snern1h@typepad.com,Genderfluid,732-79-8205,2025-02-11
35,Kermit,McGeraghty,kmcgeraghtyy@sees...,Male,262-21-3682,2025-02-05
40,Franklyn,Kedwell,fkedwell13@behanc...,Male,641-02-3450,2025-02-05


In [0]:
'''
Creating hash to keep the consolidate of the row
when the new data with same id but different hash means there will be change in the hash 
so that data should be inserted as new row

!!Fetch user_df from database
'''
user_df = user_df.withColumn("row_hash",F.sha1(F.concat_ws(*user_df.columns))) 
new_user_df = new_user_df.withColumn("row_hash",F.sha1(F.concat_ws(*new_user_df.columns)))

In [0]:
#find the changed data
changed_data = new_user_df.join(user_df,on="id", how="left").filter(new_user_df["row_hash"] != user_df["row_hash"]).select(new_user_df["*"])
#find the unchanged data
unchanged_data = user_df.join(changed_data, on="id", how="left_anti")
#find the old data that has changes in incoming data
updated_old_data = user_df.join(changed_data, on="id", how="inner").select(user_df['*'])

#set the new data to active
changed_data = changed_data.withColumn("end_date", F.lit(None)).withColumn("is_current",F.lit(True))
#set the unchanged data to active
unchanged_data = unchanged_data.withColumn("end_date", F.lit(None)).withColumn("is_current",F.lit(True))
#set the changed data to in-active
updated_old_data = updated_old_data.withColumn("end_date", F.current_date()).withColumn("is_current", F.lit(False))

In [0]:
changed_data.display()

id,first_name,last_name,email,gender,ssn,inserted_date,row_hash,end_date,is_current
18,Tani,Swannick,tswannickh@studio...,Female,773-49-1332,2025-02-20,878ac81171a5058d17e8ba9471af27d35faab7d8,null,true
35,Kermit,McGeraghty,kmcgeraghtyy@sees...,Male,262-21-3682,2025-02-05,09c7f30f571eb3f6ec3b4473cd6e8b748f5c1a8a,null,true
40,Franklyn,Kedwell,fkedwell13@behanc...,Male,641-02-3450,2025-02-05,a60800420d4e78d92c0340cbc6a33684e666e7ee,null,true
49,Marci,Quelch,mquelch1c@aol.com,Female,598-09-0176,2025-02-15,39ae30b70eed7b25ed8178522cd207725ff20fdd,null,true
54,Sara,Nern,snern1h@typepad.com,Genderfluid,732-79-8205,2025-02-11,70d24656bd5faabc88f3ebb95b7e3f915613532c,null,true
59,Ketti,Stronghill,kstronghill1m@vk.com,Non-binary,330-21-6946,2025-02-12,4c53eade30289d81951b8d7f4126b2bd351dc641,null,true
98,Henry,Bygate,hbygate2p@arstech...,Male,863-59-9997,2025-02-17,4f81e28004234e32daf688f951a3a781feabf205,null,true


In [0]:
final_df = unchanged_data.union(updated_old_data).union(changed_data)

In [0]:
final_df.orderBy("id").display()    

id,first_name,last_name,email,gender,ssn,inserted_date,row_hash,end_date,is_current
1,Odetta,Aylett,oaylett0@twitpic.com,Female,203-14-6684,2024-11-22,7a25c2e7d105f3243fe764ca95938abf8af94019,null,true
2,Antonietta,McGourty,amcgourty1@nih.gov,Female,863-90-1571,2024-02-23,142787f3a7065a6a945b6ac440567012ba16369e,null,true
3,Javier,Branthwaite,jbranthwaite2@google.com,Male,706-58-8773,2024-09-24,6c8792f9027bc79d3f7e1a9f44335a76524b28f5,null,true
4,Sydelle,Barenski,sbarenski3@youku.com,Female,899-60-7100,2024-09-17,f52bff9c830f26d45b18f8058de362631360be83,null,true
5,Emlyn,Ferandez,eferandez4@t-online.de,Male,685-46-8896,2024-06-01,ba344451dc29e3eb2a23e34e9b930eb6e4653be6,null,true
6,Dino,Kynman,dkynman5@wiley.com,Male,758-28-5682,2024-04-24,3996424fc5f1e65b376d6de2e93b8b44024f9dbf,null,true
7,Joelynn,Randerson,jranderson6@nyu.edu,Female,231-04-1761,2024-12-14,dad7bf85d6a7b8aedbe19340440e8a543c153d75,null,true
8,Yurik,Dunster,ydunster7@amazon.co.uk,Genderqueer,807-48-2811,2024-10-29,e560a137af5bbde303023a884508855229feba09,null,true
9,Diann,Beaver,dbeaver8@psu.edu,Female,759-05-6633,2024-10-23,ba24f780777863f8aef4d011a04c4638362b4157,null,true
10,Winston,Puffett,wpuffett9@over-blog.com,Male,530-42-1726,2024-06-03,d4b0dd14f05f044410412f2d9c52b7b2d54010ee,null,true


In [0]:
final_df.createOrReplaceTempView("user_table")

In [0]:
%sql
SELECT
    * 
FROM user_table


id,first_name,last_name,email,gender,ssn,inserted_date,row_hash,end_date,is_current
1,Odetta,Aylett,oaylett0@twitpic.com,Female,203-14-6684,2024-11-22,7a25c2e7d105f3243fe764ca95938abf8af94019,null,true
2,Antonietta,McGourty,amcgourty1@nih.gov,Female,863-90-1571,2024-02-23,142787f3a7065a6a945b6ac440567012ba16369e,null,true
3,Javier,Branthwaite,jbranthwaite2@google.com,Male,706-58-8773,2024-09-24,6c8792f9027bc79d3f7e1a9f44335a76524b28f5,null,true
4,Sydelle,Barenski,sbarenski3@youku.com,Female,899-60-7100,2024-09-17,f52bff9c830f26d45b18f8058de362631360be83,null,true
5,Emlyn,Ferandez,eferandez4@t-online.de,Male,685-46-8896,2024-06-01,ba344451dc29e3eb2a23e34e9b930eb6e4653be6,null,true
6,Dino,Kynman,dkynman5@wiley.com,Male,758-28-5682,2024-04-24,3996424fc5f1e65b376d6de2e93b8b44024f9dbf,null,true
7,Joelynn,Randerson,jranderson6@nyu.edu,Female,231-04-1761,2024-12-14,dad7bf85d6a7b8aedbe19340440e8a543c153d75,null,true
8,Yurik,Dunster,ydunster7@amazon.co.uk,Genderqueer,807-48-2811,2024-10-29,e560a137af5bbde303023a884508855229feba09,null,true
9,Diann,Beaver,dbeaver8@psu.edu,Female,759-05-6633,2024-10-23,ba24f780777863f8aef4d011a04c4638362b4157,null,true
10,Winston,Puffett,wpuffett9@over-blog.com,Male,530-42-1726,2024-06-03,d4b0dd14f05f044410412f2d9c52b7b2d54010ee,null,true
